In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.sequence import village_bootstrap
from uavloc.track import load_runs

# Params

In [ ]:
EXP = "027-sequence-filters"
BOOTSTRAP_RESAMPLES = 10_000
SEED = 42
NORMAL = {"ins": "027.1", "replace": "027.2", "gated": "027.3", "particles": "027.4", "hypotheses": "027.5"}
COLD = {"ins": "027.6", "replace": "027.7", "gated": "027.8", "particles": "027.9", "hypotheses": "027.10"}
HEAD = {"replace": "027.11", "particles": "027.12", "hypotheses": "027.13"}
BAR_RATIO = 0.7
MAX_FIX_MS = 1000.0
RECOVERED_M = 10.0
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]

# Data

In [ ]:
runs = {r["run"]: r for r in load_runs(EXP)}
flights = {run: pd.read_csv(Path("artifacts") / f"flights-{run}.csv") for run in runs}
fixes = {run: pd.read_csv(Path("artifacts") / f"fixes-{run}.csv") for run in runs}
sorted(runs)

In [ ]:
def run_row(run: str) -> dict:
    m, table = runs[run]["metrics"], flights[run]
    return {"run": run, "arm": runs[run]["config"]["arm"], "scenario": runs[run]["config"]["scenario"],
            "likelihood": runs[run]["config"].get("likelihood", "weighted-ncc"),
            "mean_end_m": m["mean_end_error_m"], "median_end_m": m["median_end_error_m"],
            "p90_end_m": float(table["end_error_m"].quantile(0.9)), "mean_error_m": m["mean_error_m"],
            "diverged": m["divergence_share"], "recovered_10m": m["recovered_share"],
            "in_window": m["truth_in_window_share"], "in_ellipse": m["truth_in_ellipse_share"],
            "skipped": m["skipped_share"], "median_fix_ms": m["median_fix_ms"]}

In [ ]:
summary = pd.DataFrame([run_row(run) for run in sorted(runs, key=lambda r: int(r.split(".")[1]))])
summary.round(3)

# Bar

In [ ]:
def paired(a: str, b: str, column: str = "end_error_m", villages: list[str] | None = None) -> dict:
    fa, fb = flights[a], flights[b]
    if villages is not None:
        fa, fb = fa[fa["village"].isin(villages)], fb[fb["village"].isin(villages)]
    delta, low, high = village_bootstrap(fa, fb, column, BOOTSTRAP_RESAMPLES, SEED)
    return {"a": a, "b": b, "column": column, "flights": len(fa), "mean_a": float(fa[column].astype(float).mean()),
            "mean_b": float(fb[column].astype(float).mean()), "delta": delta, "ci_low": low, "ci_high": high}

In [ ]:
bar_filter = runs[NORMAL["particles"]]["config"]["bar_filter"]
chosen = NORMAL[bar_filter]
baseline = min((NORMAL["replace"], NORMAL["gated"]), key=lambda r: runs[r]["metrics"]["mean_end_error_m"])
bar_pair = paired(chosen, baseline)
bar = {
    "bar_filter": bar_filter, "chosen_run": chosen, "baseline_run": baseline,
    "ratio": runs[chosen]["metrics"]["mean_end_error_m"] / runs[baseline]["metrics"]["mean_end_error_m"],
    "difference": bar_pair,
    "bar1_ratio_and_ci": bool(runs[chosen]["metrics"]["mean_end_error_m"] <= BAR_RATIO * runs[baseline]["metrics"]["mean_end_error_m"]
                              and bar_pair["ci_high"] < 0),
    "bar2_divergence": bool(runs[chosen]["metrics"]["divergence_share"] <= runs[baseline]["metrics"]["divergence_share"]),
    "bar3_time": bool(runs[chosen]["metrics"]["median_fix_ms"] <= MAX_FIX_MS),
}
bar["met"] = bar["bar1_ratio_and_ci"] and bar["bar2_divergence"] and bar["bar3_time"]
bar

# Secondary

In [ ]:
secondary = pd.DataFrame([
    {"comparison": "A vs B, normal", **paired(NORMAL["particles"], NORMAL["hypotheses"])},
    {"comparison": "A vs B, cold", **paired(COLD["particles"], COLD["hypotheses"])},
    {"comparison": "A vs INS, cold", **paired(COLD["particles"], COLD["ins"])},
    {"comparison": "B vs INS, cold", **paired(COLD["hypotheses"], COLD["ins"])},
    {"comparison": "A vs replace, normal", **paired(NORMAL["particles"], NORMAL["replace"])},
    {"comparison": "A vs gated, normal", **paired(NORMAL["particles"], NORMAL["gated"])},
    {"comparison": "B vs best baseline, normal", **paired(NORMAL["hypotheses"], baseline)},
])
secondary.round(3)

In [ ]:
present_head = {arm: run for arm, run in HEAD.items() if run in runs}
head = pd.DataFrame([{"comparison": f"head vs NCC, {arm}, clean flights", **paired(run, NORMAL[arm], villages=HEAD_CLEAN_VILLAGES)}
                     for arm, run in present_head.items()]
                    + [{"comparison": f"head vs NCC, {arm}, all test flights", **paired(run, NORMAL[arm])}
                       for arm, run in present_head.items()])
head.round(3)

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_end_error_cdf({arm: flights[run]["end_error_m"] for arm, run in NORMAL.items()}, axes[0])
axes[0].set_title("normal start, 94 test flights")
plot_end_error_cdf({arm: flights[run]["end_error_m"] for arm, run in COLD.items()}, axes[1])
axes[1].set_title("cold start (40 m), 94 test flights")
fig.tight_layout()

In [ ]:
fig_window, ax = plt.subplots(figsize=(7, 3.5))
for arm in ("particles", "hypotheses"):
    f = fixes[NORMAL[arm]]
    ax.hist(f["error_before_m"], bins=np.logspace(-1, 2.5, 40), histtype="step", label=f"{arm}: error before fix")
ax.set(xscale="log", xlabel="m", ylabel="fixes", title="normal start: error before each fix")
ax.legend()
fig_window.tight_layout()

In [ ]:
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/end-error-cdf.png", dpi=110)
fig_window.savefig("figures/error-before-fix.png", dpi=110)
Path("artifacts/analysis.json").write_text(json.dumps({"summary": summary.to_dict("records"), "bar": bar,
                                                       "secondary": secondary.to_dict("records"),
                                                       "head": head.to_dict("records")}, indent=1, default=float))